# 2 Exploration

In [ ]:
import pandas as pd
from pathlib import Path 

destination = Path(r"filepath\csv_basic_clean")

companies=pd.read_csv(destination / "companies_bclean.csv")
index=pd.read_csv(destination / "index_bclean.csv")
stocks=pd.read_csv(destination / "stocks_bclean.csv")


## 2.1. Coverage

In [2]:
print("Companies table","\nnumber of companies:",len(companies), "\nnumber of sectors:", companies["sector"].nunique())

print("Stocks table","\nearliest date:",stocks["date"].min(), "\nlatest date:", stocks["date"].max(), "\nnumber of rows:",
      len(stocks),"\nnumber of stocks:",stocks["symbol"].nunique())

print("Index table", "\nearliest date:", index['date'].min(), "\nlatest date:", index['date'].max(), "\nnumber of rows:", 
      len(index))

"""
Companies table 
number of companies: 502 
number of sectors: 11

Stocks table 
earliest date: 2010-01-04 
latest date: 2024-12-20 
number of rows: 1891536 
number of stocks: 502

Index table 
earliest date: 2014-12-22 
latest date: 2024-12-20 
number of rows: 2517
"""

Companies table 
number of companies: 502 
number of sectors: 11
Stocks table 
earliest date: 2010-01-04 
latest date: 2024-12-20 
number of rows: 1891536 
number of stocks: 502
Index table 
earliest date: 2014-12-22 
latest date: 2024-12-20 
number of rows: 2517


'\nCompanies table \nnumber of companies: 502 \nnumber of sectors: 11\n\nStocks table \nearliest date: 2010-01-04 \nlatest date: 2024-12-20 \nnumber of rows: 1891536 \nnumber of stocks: 502\n\nIndex table \nearliest date: 2014-12-22 \nlatest date: 2024-12-20 \nnumber of rows: 2517\n'

1.1. Why 502 and not 500 stocks if it's the sp500?

In [ ]:
#Checking which companies have more than one symbol:
dual_class=companies.groupby("long_name")["symbol"].agg(n_symbols='nunique', symbols=lambda s: ', '.join(s.dropna().astype(str)))

dual_class[dual_class["n_symbols"]>1]

,n_symbols,symbols
long_name,,
Alphabet Inc.,2,"GOOGL, GOOG"
Fox Corporation,2,"FOXA, FOX"
News Corporation,2,"NWS, NWSA"


1.2. Confirm that every tickers has the same number of rows:

In [ ]:
print(stocks.groupby('symbol')['date'].nunique().value_counts().rename_axis('n_dates').rename('n_symbols').reset_index())

   n_dates  n_symbols
0     3768        502


'another way:\n(stocks.groupby("symbol")["date"]\n    .nunique()\n    .value_counts()\n    .sort_index()\n    .rename_axis("n_days")\n    .reset_index(name="n_symbols"))\n'

## 2.2 - Duplicates

In [5]:
print("Duplicate index dates:", index['date'].duplicated().sum())
print("Duplicate (date, symbol) in stocks:", stocks.duplicated(subset=['date', 'symbol']).sum())
print("Duplicate symbols in companies:", companies['symbol'].duplicated().sum())

Duplicate index dates: 0
Duplicate (date, symbol) in stocks: 0
Duplicate symbols in companies: 0


## 2.3 - Missing values

2.3.1 - Any column nulls in the tables?

In [6]:
"COMPANIES TABLE:"

companies.isnull().sum()
#no missing values in relevant columns.

exchange                  0
symbol                    0
short_name                0
long_name                 0
sector                    0
industry                  0
current_price             0
marketcap                 0
ebitda                   29
revenue_growth            3
city                      0
state                    20
country                   0
fulltime_employees        9
long_business_summary     0
weight                    0
dtype: int64

In [ ]:
#STOCKS TABLE missing values:

pd.DataFrame({
    "n_nulls":stocks.isnull().sum(),
    "nulls %":(stocks.isnull().sum()/len(stocks)*100).round(2)
})
  

,n_nulls,nulls %
date,0,0.00
symbol,0,0.00
adj_close,100475,5.31
close,100475,5.31
high,100475,5.31
low,100475,5.31
open,100475,5.31
volume,100475,5.31


In [8]:
"INDEX TABLE:"

index.isnull().sum()

#no missing values

date     0
sp500    0
dtype: int64

2.3.2 - Nulls further investigation:

In [9]:
stocks.groupby("symbol")["adj_close"].apply(lambda s: s.notna().sum()==0).value_counts()

adj_close
False    502
Name: count, dtype: int64

In [ ]:
#Any symbol with adj_close = Null rows?

((stocks.groupby('symbol')['adj_close'].apply(lambda s: s.notna().sum()))==0).value_counts().rename_axis("all_null")



all_null
False    502
Name: count, dtype: int64

In [ ]:
#Find the first and last real price data for each symbol:

valid_only=stocks[stocks["adj_close"].notna()]
#find the minimum and maximum date (first non null value and last non null value):
first_real=valid_only.groupby("symbol")["date"].min()
last_real=valid_only.groupby("symbol")["date"].max()

stocks['first_real_date'] = stocks['symbol'].map(first_real)

stocks['last_real_date'] = stocks['symbol'].map(last_real)   

is_null = stocks["adj_close"].isna()

print("nulls before:", (is_null & (stocks["date"]<stocks["first_real_date"])).sum())
print("nulls between:", (is_null & (stocks["date"].between(stocks["first_real_date"],stocks["last_real_date"]))).sum())
print("nulls after:", (is_null & (stocks["date"]>stocks["last_real_date"])).sum())

nulls before: 100475
nulls between: 0
nulls after: 0


In [ ]:
#Find if there are symbols or rows with only null values in all value columns (adj_close, close, high, low, open, volume)
value_columns=["adj_close","close", "high", 'low', 'open', 'volume']

nulls=stocks[value_columns].isna()

all_nulls=nulls.all(axis=1).sum()
partial_nulls=nulls.any(axis=1) & ~nulls.all(axis=1)

print("All prices null:", all_nulls.sum(), "\nPartial nulls:", partial_nulls.sum())

#All prices null: 100475 
#Partial nulls: 0


All prices null: 100475 
Partial nulls: 0


## 2.4 Bad values

In [13]:
"STOCKS TABLE non realistic values (negative values):"

(stocks[
    (stocks["adj_close"] <= 0)
    | (stocks["close"] <= 0)
    | (stocks["high"] <= 0)
    | (stocks["low"] <= 0)
    | (stocks["open"] <= 0)
    | (stocks["volume"] < 0)
]
)
#no results

,date,symbol,adj_close,close,high,low,open,volume,first_real_date,last_real_date


In [ ]:
#Check stats of the columns adj_close and volume:
stocks[["adj_close","volume"]].describe().round(2)

,adj_close,volume
count,1791061.00,1.791061e+06
mean,98.09,7.022543e+06
std,218.34,3.299791e+07
min,0.11,0.000000e+00
25%,28.83,1.014100e+06
50%,53.92,2.217500e+06
75%,104.14,5.085700e+06
max,9924.40,3.692928e+09


Check the zero volume minimum result:

In [15]:
flat = stocks[
    (stocks['volume'] == 0) & (stocks['open'] == stocks['high']) &
    (stocks['high'] == stocks['low']) & (stocks['low'] == stocks['close'])
]

len(flat)

#Result: 1406 wos with 0 volume and flat values

1406

In [16]:
near_start = (flat['date'] <= flat['first_real_date']).sum()
near_end = (flat['date'] >= flat['last_real_date']).sum()
mid_series = len(flat) - near_start - near_end

print("near_start:", near_start, "| mid_series:", mid_series, "| near_end:", near_end)

near_start: 2 | mid_series: 1404 | near_end: 0


In [17]:
#Which dates:
print(flat['date'].value_counts().head(15))
#Which symbols:
print(flat['symbol'].value_counts().head(15))

date
2018-06-08    3
2015-01-02    2
2017-01-17    2
2019-11-05    2
2016-01-06    2
2017-12-18    2
2016-07-21    2
2016-11-25    2
2016-12-29    1
2017-01-10    1
2017-01-09    1
2017-01-06    1
2017-01-05    1
2017-01-04    1
2017-01-03    1
Name: count, dtype: int64
symbol
AMCR    1371
CHTR      18
PFG        2
AMD        1
GDDY       1
WTW        1
VST        1
VRTX       1
SBAC       1
DOC        1
GEN        1
ERIE       1
DXCM       1
CCI        1
CHD        1
Name: count, dtype: int64


##### Confirm the max and min values:

In [ ]:
#5 largest values of adj_close:
print('5 largest:\n',stocks.dropna(subset=['adj_close']).nlargest(5, 'adj_close')[['symbol', 'date', 'adj_close']])
print('5 smallest:\n',stocks.dropna(subset=['adj_close']).nsmallest(5,'adj_close')[['symbol', 'date', 'adj_close']])


5 largest:
         symbol        date    adj_close
1314987    NVR  2024-10-18  9924.400391
1314974    NVR  2024-10-01  9892.299805
1314985    NVR  2024-10-16  9875.580078
1314973    NVR  2024-09-30  9811.799805
1314986    NVR  2024-10-17  9773.580078
5 smallest:
        symbol        date  adj_close
271322   AVGO  2010-02-10   0.112745
271321   AVGO  2010-02-09   0.114020
271320   AVGO  2010-02-08   0.114154
271323   AVGO  2010-02-11   0.114959
271319   AVGO  2010-02-05   0.115429


In [ ]:
#STOCKS TABLE high lower than the other columns and low higher than the other columns:

(stocks[
    (stocks["high"] < stocks["low"])
    | (stocks["high"] < stocks["close"])
    | (stocks["high"] < stocks["open"])
    | (stocks["low"] > stocks["close"])
    | (stocks["low"] > stocks["open"])
])

#One data+symbol that has a lower > open. The difference is minimal though (0.24).
#These values were confirmed using yahoo finance, so it was not a data saving problem.

,date,symbol,adj_close,close,high,low,open,volume,first_real_date,last_real_date
903405,2021-05-05,HUBB,185.76297,198.130005,198.639999,196.210007,195.979996,127234.0,2010-01-04,2024-12-20


In [ ]:
#used this to get the may 2021 history of that stock and to cross check with other online sources:
stocks[
    (stocks["symbol"]=="HUBB") &
    (stocks["date"]>="2021-05-01") &
    (stocks["date"]<"2021-06-01")
]


,date,symbol,adj_close,close,high,low,open,volume,first_real_date,last_real_date
903403,2021-05-03,HUBB,181.534470,193.619995,195.369995,192.279999,193.440002,264291.0,2010-01-04,2024-12-20
903404,2021-05-04,HUBB,183.287766,195.490005,196.000000,190.710007,192.210007,137031.0,2010-01-04,2024-12-20
903405,2021-05-05,HUBB,185.762970,198.130005,198.639999,196.210007,195.979996,127234.0,2010-01-04,2024-12-20
903406,2021-05-06,HUBB,187.309952,199.779999,199.970001,196.009995,198.039993,172445.0,2010-01-04,2024-12-20
903407,2021-05-07,HUBB,186.747437,199.179993,201.029999,198.169998,198.940002,210932.0,2010-01-04,2024-12-20
903408,2021-05-10,HUBB,185.162903,197.490005,201.059998,197.250000,199.880005,158835.0,2010-01-04,2024-12-20
903409,2021-05-11,HUBB,183.606537,195.830002,197.820007,194.539993,195.309998,191589.0,2010-01-04,2024-12-20
903410,2021-05-12,HUBB,176.996582,188.779999,196.000000,188.690002,195.360001,199427.0,2010-01-04,2024-12-20
903411,2021-05-13,HUBB,179.771851,191.740005,192.860001,188.414993,188.899994,350094.0,2010-01-04,2024-12-20
903412,2021-05-14,HUBB,181.975113,194.089996,194.460007,191.410004,193.360001,254056.0,2010-01-04,2024-12-20


I found in yahoo finance that the low is in fact higher than the open. So the anomaly is not just in this dataset.
Can't explain why that is.

## 2.6 Cross table check:

In [ ]:
# set(stocks['symbol'].unique()) we don't need the unique here because sets don't allow duplicates, deleting them when converting
print((set(stocks['symbol'])) - (set(companies['symbol'])))
print((set(companies['symbol'])) - (set(stocks['symbol'])))
#subtraction of two sets gives the different elements in both sets. it's not an arithmetic subtraction. 

#result were empty sets so all symbols in stocks exist in companies, and all symbols in companies exist in stocks


set()
set()


#### Saving cleaned tables:

No need to save anything from this section. We have created a few new columns, but those won't be used in the rest of this project.

FINDINGS:

- 1,891,536 rows, 502 tickers, 3768 calendar rows each
- 502 vs. 500 explained by dual-class shares (GOOG/GOOGL, FOX/FOXA, NWS/NWSA)
- 0 duplicates, 0 cross-table mismatches
- 1 inconsistent values row (HUBB, 2021-05-05) - negligible
- 5.3% of stock rows fully null across all 6 value fields, 0 partial-null rows, all nulls fall before each ticker's first real price (none between/after)
- 1,406 flat/zero-volume rows (0.07%), mostly mid-series, 97.5% concentrated in one ticker (AMCR) - wont be removed
